# BCI Transfer Learning for Motor Imagery

This notebook runs the full transfer learning pipeline on **Google Colab** or **Kaggle** with GPU support.

**Supported Datasets:**
- BCI Competition IV 2a (4-class, 9 subjects)
- BCI Competition IV 2b (2-class, 9 subjects)
- PhysioNet Motor Imagery (4-class, 109 subjects – auto-downloads)

**Models:** EEGNet, ShallowConvNet, DeepConvNet, **EEG-Inception**

**TL Strategies:** Fine-Tuning, Feature Extraction, MMD/CORAL, DANN, Progressive, Multi-Source

## 1. Detect Platform & Install Dependencies

In [ ]:
# Detect platform
import os, sys

IS_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
IS_KAGGLE = os.path.exists('/kaggle')
PLATFORM = 'Kaggle' if IS_KAGGLE else ('Colab' if IS_COLAB else 'Local')
print(f'Platform: {PLATFORM}')

# Check GPU
!nvidia-smi

# Install dependencies (pre-installed on both platforms, but ensure versions)
if IS_COLAB:
    !pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
    !pip install -q mne scipy scikit-learn matplotlib
elif IS_KAGGLE:
    # Kaggle has torch pre-installed; just add MNE
    !pip install -q mne
else:
    !pip install -q torch mne scipy scikit-learn matplotlib

## 2. Upload Project Code

| Platform | Recommended Method |
|----------|--------------------|
| **Colab** | Upload `.py` files or mount Google Drive |
| **Kaggle** | Add as a Kaggle Dataset (upload `.py` files as a dataset, then copy) |

In [ ]:
# ──── Upload project code (platform-aware) ────
import os, shutil

if IS_KAGGLE:
    # On Kaggle: Upload all .py files as a Kaggle Dataset named "bci-transfer-learning-code"
    # Then they appear at /kaggle/input/bci-transfer-learning-code/
    WORK_DIR = '/kaggle/working'
    CODE_INPUT = '/kaggle/input/bci-transfer-learning-code'  # <── your dataset name

    if os.path.exists(CODE_INPUT):
        # Copy .py files from input (read-only) to working (writable)
        for f in os.listdir(CODE_INPUT):
            if f.endswith('.py'):
                shutil.copy2(os.path.join(CODE_INPUT, f), os.path.join(WORK_DIR, f))
                print(f'  Copied: {f}')
        os.chdir(WORK_DIR)
    else:
        print(f'⚠ Dataset not found at {CODE_INPUT}')
        print('  → Go to "Add Data" → "Upload" → create dataset "bci-transfer-learning-code"')
        print('  → Upload: config.py, preprocessing.py, datasets.py, models.py,')
        print('    transfer_learning.py, trainer.py, main.py')

elif IS_COLAB:
    # On Colab: Upload files manually
    from google.colab import files
    os.makedirs('BCI_Transfer_Learning', exist_ok=True)
    os.chdir('BCI_Transfer_Learning')
    print('Upload all .py files:')
    uploaded = files.upload()
    print(f'\nUploaded {len(uploaded)} files: {list(uploaded.keys())}')

print(f'Working directory: {os.getcwd()}')

In [ ]:
# ──── Persistent storage (Google Drive / Kaggle Output) ────
import os

if IS_COLAB:
    # Mount Google Drive for persistent data & checkpoints
    from google.colab import drive
    drive.mount('/content/drive')

    PROJECT_DIR = '/content/BCI_Transfer_Learning'
    DRIVE_DATA  = '/content/drive/MyDrive/BCI_Data'  # <── change to your folder
    os.makedirs(PROJECT_DIR, exist_ok=True)
    os.makedirs(DRIVE_DATA, exist_ok=True)

    data_link = os.path.join(PROJECT_DIR, 'data')
    if not os.path.exists(data_link):
        os.symlink(DRIVE_DATA, data_link)
    os.chdir(PROJECT_DIR)

elif IS_KAGGLE:
    # Kaggle: input datasets are at /kaggle/input/, output at /kaggle/working/
    os.chdir('/kaggle/working')
    os.makedirs('data', exist_ok=True)

    # If you uploaded BCI IV 2a as a Kaggle dataset named "bci-iv-2a":
    BCI2A_INPUT = '/kaggle/input/bci-iv-2a'       # <── your dataset name
    BCI2B_INPUT = '/kaggle/input/bci-iv-2b'       # <── your dataset name

    for src, dst in [(BCI2A_INPUT, 'data/bci_iv_2a'), (BCI2B_INPUT, 'data/bci_iv_2b')]:
        if os.path.exists(src) and not os.path.exists(dst):
            os.symlink(src, dst)
            print(f'Linked: {src} → {dst}')

print(f'Working directory: {os.getcwd()}')
print(f'Data directory contents: {os.listdir("data") if os.path.exists("data") else "not created"}')

## 3. Dataset Setup

### How to get datasets onto each platform

| Dataset | Colab | Kaggle |
|---------|-------|--------|
| **PhysioNet MI** | Auto-downloads via MNE (run cell below) | Auto-downloads via MNE, OR add [this Kaggle dataset](https://www.kaggle.com/datasets/andrewmvd/eeg-motor-movementimagery-dataset) |
| **BCI IV 2a** | Upload `.gdf` files or put in Google Drive | Create Kaggle Dataset → upload `.gdf` + `.mat` files → Add Data |
| **BCI IV 2b** | Same as 2a | Same as 2a |

In [ ]:
# ──── PhysioNet MI: Auto-download via MNE (works on BOTH platforms) ────
import os
os.makedirs('data/physionet_mi', exist_ok=True)

# On Kaggle, you can ALSO use the pre-uploaded dataset:
# https://www.kaggle.com/datasets/andrewmvd/eeg-motor-movementimagery-dataset
KAGGLE_PHYSIONET = '/kaggle/input/eeg-motor-movementimagery-dataset'

if IS_KAGGLE and os.path.exists(KAGGLE_PHYSIONET):
    # Link Kaggle's pre-uploaded PhysioNet dataset
    if not os.path.islink('data/physionet_mi') and not os.listdir('data/physionet_mi'):
        os.rmdir('data/physionet_mi')
        os.symlink(KAGGLE_PHYSIONET, 'data/physionet_mi')
    print(f'Using Kaggle dataset: {KAGGLE_PHYSIONET}')
    print(f'Files: {os.listdir("data/physionet_mi")[:10]}...')
else:
    # Auto-download via MNE (Colab or Kaggle without pre-uploaded data)
    import mne
    from mne.datasets import eegbci

    for subject_id in range(1, 10):
        print(f'Downloading subject {subject_id}...')
        for run in [4, 6, 8, 10, 12, 14]:  # Motor imagery runs
            try:
                eegbci.load_data(subject_id, [run], path='data/physionet_mi')
            except Exception as e:
                print(f'  Run {run}: {e}')

print('\nPhysioNet MI data ready!')

In [ ]:
# ──── BCI Competition IV 2a: Upload GDF files ────
# Download from: http://bnci-horizon-2020.eu/database/data-sets
# Or: https://www.bbci.de/competition/iv/#datasets
#
# You need files: A01T.gdf, A01E.gdf, ..., A09T.gdf, A09E.gdf
# Plus label files: A01E.mat, ..., A09E.mat (for evaluation sessions)

import os
os.makedirs('data/bci_iv_2a', exist_ok=True)

# ---- Method 1: Upload from local machine ----
from google.colab import files
print('Upload all .gdf and .mat files for BCI IV 2a:')
uploaded = files.upload()
for fname, content in uploaded.items():
    with open(f'data/bci_iv_2a/{fname}', 'wb') as f:
        f.write(content)
    print(f'  Saved: data/bci_iv_2a/{fname}')

# ---- Method 2: From Google Drive (if already uploaded there) ----
# Run the Drive mount cell above, then:
# !cp /content/drive/MyDrive/BCI_Data/bci_iv_2a/*.gdf data/bci_iv_2a/
# !cp /content/drive/MyDrive/BCI_Data/bci_iv_2a/*.mat data/bci_iv_2a/

# ---- Method 3: Direct download via gdown (if on Drive link) ----
# !pip install -q gdown
# !gdown --fuzzy 'https://drive.google.com/file/d/YOUR_FILE_ID' -O data/bci_iv_2a/

print(f"\nFiles in data/bci_iv_2a/: {os.listdir('data/bci_iv_2a')}")

In [ ]:
# ──── BCI Competition IV 2b: Upload GDF files ────
import os
os.makedirs('data/bci_iv_2b', exist_ok=True)

# Same upload process as 2a
from google.colab import files
print('Upload .gdf files for BCI IV 2b:')
uploaded = files.upload()
for fname, content in uploaded.items():
    with open(f'data/bci_iv_2b/{fname}', 'wb') as f:
        f.write(content)

print(f"\nFiles in data/bci_iv_2b/: {os.listdir('data/bci_iv_2b')}")

## 4. Verify Setup

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

import mne
print(f'MNE: {mne.__version__}')

# Verify project files
import os
required = ['config.py', 'preprocessing.py', 'datasets.py', 'models.py',
            'transfer_learning.py', 'trainer.py', 'main.py']
for f in required:
    status = '✓' if os.path.exists(f) else '✗ MISSING'
    print(f'  {status} {f}')

# Verify data directories
for d in ['data/bci_iv_2a', 'data/bci_iv_2b', 'data/physionet_mi']:
    if os.path.exists(d):
        n_files = len(os.listdir(d))
        print(f'  ✓ {d}/ ({n_files} files)')
    else:
        print(f'  - {d}/ (not set up)')

## 5. Run Experiments

### Quick Test: PhysioNet MI with EEGNet

In [ ]:
# Test with a small experiment first (2 source subjects, fast epochs)
!python main.py \
    --dataset physionet \
    --model eegnet \
    --strategy fine_tune \
    --target 5 \
    --source 1 2 3 4 \
    --n-cal 10 \
    --epochs-pretrain 30 \
    --epochs-finetune 20 \
    --device cuda \
    --no-ica

### Run with EEG-Inception

In [ ]:
!python main.py \
    --dataset physionet \
    --model eeginception \
    --strategy fine_tune \
    --target 5 \
    --source 1 2 3 4 \
    --n-cal 10 \
    --epochs-pretrain 30 \
    --epochs-finetune 20 \
    --device cuda \
    --no-ica

### BCI Competition IV 2a Experiment

In [ ]:
# Full experiment on BCI IV 2a (requires uploaded GDF files)
!python main.py \
    --dataset bci_iv_2a \
    --model eegnet \
    --strategy fine_tune \
    --target 9 \
    --n-cal 10 \
    --epochs-pretrain 200 \
    --epochs-finetune 100 \
    --device cuda

### Compare All TL Strategies

In [ ]:
# Compare all 5 TL strategies on the same target subject
!python main.py \
    --compare-all \
    --dataset physionet \
    --model eeginception \
    --target 5 \
    --source 1 2 3 4 6 7 8 9 \
    --epochs-pretrain 100 \
    --epochs-finetune 50 \
    --device cuda \
    --no-ica

### Compare All Models on Same Strategy

In [ ]:
# Compare all 4 models with fine-tuning
import subprocess, sys

models = ['eegnet', 'shallowconvnet', 'deepconvnet', 'eeginception']
results = {}

for model in models:
    print(f'\n{"="*60}')
    print(f'Running: {model}')
    print(f'{"="*60}')
    !python main.py \
        --dataset physionet \
        --model {model} \
        --strategy fine_tune \
        --target 5 \
        --source 1 2 3 4 \
        --epochs-pretrain 50 \
        --epochs-finetune 30 \
        --device cuda \
        --no-ica

## 6. Run from Python API (Alternative to CLI)

In [ ]:
import sys
sys.path.insert(0, '.')

import torch
import numpy as np
from config import PreprocessingConfig, DataConfig, ModelConfig, TransferConfig, TrainConfig
from models import build_model
from datasets import create_source_target_loaders
from trainer import run_transfer_pipeline
from main import set_seed

# ── Configuration ──
preproc = PreprocessingConfig(use_ica=False)  # ICA is slow; disable for quick tests
data_cfg = DataConfig(physionet_path='data/physionet_mi')
model_cfg = ModelConfig(model_type='eeginception')
transfer_cfg = TransferConfig(
    strategy='fine_tune',
    source_subjects=[1, 2, 3, 4, 6, 7, 8],
    target_subject=5,
    n_target_trials_per_class=10,
    use_euclidean_alignment=True,
)
train_cfg = TrainConfig(
    n_epochs_pretrain=100,
    n_epochs_finetune=50,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    batch_size=64,
)

set_seed(42)

# ── Load Data ──
n_channels, n_classes = 64, 4  # PhysioNet
n_samples = int((preproc.tmax - preproc.tmin) * preproc.target_srate)

source_loader, target_train_loader, target_test_loader = create_source_target_loaders(
    dataset_name='physionet',
    data_dir='data/physionet_mi',
    source_subjects=transfer_cfg.source_subjects,
    target_subject=transfer_cfg.target_subject,
    preproc_config=preproc,
    train_config=train_cfg,
    n_target_trials_per_class=transfer_cfg.n_target_trials_per_class,
    apply_ea=transfer_cfg.use_euclidean_alignment,
)

# ── Build Model ──
model = build_model('eeginception', n_channels, n_samples, n_classes)
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

# ── Run Transfer Learning ──
results = run_transfer_pipeline(
    model=model,
    source_loader=source_loader,
    target_train_loader=target_train_loader,
    target_test_loader=target_test_loader,
    train_config=train_cfg,
    transfer_config=transfer_cfg,
    n_classes=n_classes,
)

print(f"\nFinal Accuracy: {results['metrics']['accuracy']:.4f}")
print(f"Final Kappa:    {results['metrics']['kappa']:.4f}")

## 7. Visualize Results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import ConfusionMatrixDisplay

cm = results['metrics']['confusion_matrix']
class_names = ['Left Fist', 'Right Fist', 'Both Fists', 'Both Feet']  # PhysioNet
# class_names = ['Left Hand', 'Right Hand', 'Both Feet', 'Tongue']  # BCI IV 2a

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title(f"Transfer Learning Results\n"
             f"Acc={results['metrics']['accuracy']:.3f}, "
             f"κ={results['metrics']['kappa']:.3f}")
plt.tight_layout()
plt.show()

## 8. Save Model to Google Drive

In [ ]:
import torch, os

save_dir = '/content/drive/MyDrive/BCI_Data/checkpoints'
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, 'best_model.pt')
torch.save(results['model'].state_dict(), save_path)
print(f'Model saved to {save_path}')